# Full Pipeline Run

**AI-Based Early Detection and Classification of Foot and Nail Conditions Using Transfer Learning for Rural Healthcare**

Runs the project end to end: fetch data → extract montages → preprocess → train both models.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

Every step is guarded, so re-running a cell that has already completed skips its
work instead of redoing it. If the runtime is recycled part-way through, re-run
from the top — completed steps are restored from Drive rather than recomputed.

**Storage.** All work happens on `/content`, which is fast local disk. Finished
artefacts are packed into single archives and copied to Drive, because Drive
buffers writes and loses tens of thousands of small files when the runtime dies.
Run the `save` cells when you reach them — they are what makes the work survive.

## 1. Setup

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO    = 'https://github.com/gurubasavarajharlapur-jpg/Dissertation_AI_FOOT_NAIL_DISEASE.git'
BRANCH  = 'claude/foot-nail-disease-ai-fyrdsb'
PROJECT = Path('/content/Dissertation_AI_FOOT_NAIL_DISEASE')
DRIVE   = Path('/content/drive/MyDrive/dissertation_foot_nail')

# Clone if absent, pull if present, so this cell is safe to re-run.
if (PROJECT / '.git').is_dir():
    !cd {PROJECT} && git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q --ff-only
else:
    !git clone -q --branch {BRANCH} {REPO} {PROJECT}

%cd {PROJECT}
!pip install -q -r requirements.txt
!git log --oneline -1

In [ ]:
# An earlier setup symlinked data/ into Drive, which is what lost the data.
# This copies anything still on Drive back to local disk, then removes the links.
!python src/colab_sync.py unlink
!python src/colab_sync.py status

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__, '| Keras', tf.keras.__version__)
print('GPU:', [g.name for g in gpus] if gpus else
      'NONE — set Runtime > Change runtime type > T4 GPU before the training cells')

## 2. Restore anything already saved

If a previous session got as far as preprocessing or training, this brings it
back and you can skip straight to whichever step is still outstanding.

In [ ]:
!python src/colab_sync.py restore

## 3. What raw data is present?

In [ ]:
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
RAW = PROJECT / 'data' / 'raw'

def count_images(path) -> int:
    path = Path(path)
    if not path.exists():
        return 0
    return sum(1 for f in path.rglob('*')
               if f.is_file() and f.suffix.lower() in IMAGE_EXT)

# Expected counts, from the runs that completed successfully.
EXPECTED = {
    'figshare_nail':       1635,
    'mendeley_foot':       5443,
    'ulcer_fuseg':         1370,
    'figshare_nail_tiles': 18093,
}

print(f"{'folder':<24}{'present':>9}{'expected':>10}")
have = {}
for name, expected in EXPECTED.items():
    n = count_images(RAW / name)
    have[name] = n
    flag = 'ok' if n >= expected * 0.95 else 'MISSING/PARTIAL'
    print(f'{name:<24}{n:>9}{expected:>10}   {flag}')

## 4. Fetch raw data

Each cell skips itself if the data is already there.

In [ ]:
# Figshare onychomycosis: ~2.2 GB
if have['figshare_nail'] < EXPECTED['figshare_nail'] * 0.95:
    !python src/download_data.py --source figshare
    have['figshare_nail'] = count_images(RAW / 'figshare_nail')
else:
    print('figshare_nail already present — skipping')
print('figshare_nail:', have['figshare_nail'])

In [ ]:
# FUSeg / AZH foot ulcers: ~600 MB, images only (the labels/ folders are
# segmentation masks and must never enter a classification training set).
import shutil

if have['ulcer_fuseg'] < EXPECTED['ulcer_fuseg'] * 0.95:
    !rm -rf /tmp/ulcer_repo
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/uwm-bigdata/wound-segmentation.git /tmp/ulcer_repo
    !cd /tmp/ulcer_repo && git sparse-checkout set --no-cone '/data/**/images/**'

    SRC  = Path('/tmp/ulcer_repo/data')
    DEST = RAW / 'ulcer_fuseg'
    copied = 0
    for img_dir in sorted(SRC.rglob('images')):
        parts = img_dir.relative_to(SRC).parts
        tag = ('fuseg' if 'Foot Ulcer' in parts[0] else 'medetec') + '_' + parts[-2]
        out = DEST / tag
        out.mkdir(parents=True, exist_ok=True)
        for f in img_dir.iterdir():
            if f.suffix.lower() in IMAGE_EXT:
                shutil.copy2(f, out / f.name)
                copied += 1
        print(f'  {tag:<18} {len(list(out.iterdir())):>5}')
    shutil.rmtree('/tmp/ulcer_repo', ignore_errors=True)
    have['ulcer_fuseg'] = count_images(DEST)
else:
    print('ulcer_fuseg already present — skipping')
print('ulcer_fuseg:', have['ulcer_fuseg'])

In [ ]:
# Mendeley foot images, from the zips uploaded to Drive.
import zipfile

MASKS = PROJECT / 'data' / 'mendeley_masks'

# wound_mask.zip holds SEGMENTATION MASKS, not photographs. They go outside
# data/raw so neither the inspector nor training ever sees them — a binary mask
# counted as a training image is a labelled example of nothing. They are kept
# rather than deleted, being useful for cropping to the wound later.
TARGETS = {
    'normal.zip':     RAW / 'mendeley_foot' / 'Normal',
    'wound_main.zip': RAW / 'mendeley_foot' / 'wound_main',
    'wound_mask.zip': MASKS / 'wound_mask',
}

def safe_extract(archive: Path, target: Path) -> int:
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        members = []
        for name in zf.namelist():
            p = Path(name)
            if p.is_absolute() or '..' in p.parts:
                raise RuntimeError(f'unsafe entry in {archive.name}: {name}')
            if name.startswith('__MACOSX/') or Path(name).name.startswith('._'):
                continue
            members.append(name)
        zf.extractall(target, members=members)
    return count_images(target)

if have['mendeley_foot'] < EXPECTED['mendeley_foot'] * 0.95:
    missing = [z for z in TARGETS if not (DRIVE / z).exists()]
    if missing:
        print('Missing from Drive:', missing)
        print(f'Upload them to {DRIVE}/ and re-run this cell.')
        !ls -la {DRIVE}/*.zip
    else:
        for zip_name, target in TARGETS.items():
            n = safe_extract(DRIVE / zip_name, target)
            role = 'MASKS — excluded from training' if 'mask' in zip_name else 'training data'
            print(f'  {zip_name:<16} {n:>5} images   [{role}]')
        have['mendeley_foot'] = count_images(RAW / 'mendeley_foot')
else:
    print('mendeley_foot already present — skipping')
print('mendeley_foot:', have['mendeley_foot'])

## 5. Tile the montage sheets

The Figshare A1/A2 archives are contact sheets — hundreds of nail thumbnails
tiled into one image. Each would otherwise enter training as a single example
containing hundreds of nails. Tiling them is also the only source of **healthy
nail** images, without which the model would classify a healthy nail as fungal.

In [ ]:
if have['figshare_nail_tiles'] < EXPECTED['figshare_nail_tiles'] * 0.95:
    !python src/extract_montages.py
    have['figshare_nail_tiles'] = count_images(RAW / 'figshare_nail_tiles')
else:
    print('tiles already extracted — skipping')
print('figshare_nail_tiles:', have['figshare_nail_tiles'])

In [ ]:
# Everything should be present before preprocessing runs.
print(f"{'folder':<24}{'present':>9}{'expected':>10}")
ready = True
for name, expected in EXPECTED.items():
    n = count_images(RAW / name)
    ok = n >= expected * 0.95
    ready &= ok
    print(f'{name:<24}{n:>9}{expected:>10}   {"ok" if ok else "STILL MISSING"}')
print('\nready for preprocessing:', ready)

## 6. Preprocess

In [ ]:
!python src/preprocessing.py --dry-run

In [ ]:
!python src/preprocessing.py

**Save now.** This is ~250 MB as a single archive, and it is what you would
otherwise have to rebuild after a runtime restart.

In [ ]:
!python src/colab_sync.py save --what processed

## 7. Train MobileNetV2 (primary model)

15 epochs training the head with the backbone frozen, then 25 fine-tuning the
top layers at a 100x lower learning rate, with early stopping.

In [ ]:
assert tf.config.list_physical_devices('GPU'), (
    'No GPU: set Runtime > Change runtime type > T4 GPU, then re-run from cell 1.')

!python src/train.py --model mobilenetv2

In [ ]:
!python src/colab_sync.py save --what models results

## 8. Train ResNet50 (comparison model)

In [ ]:
!python src/train.py --model resnet50

In [ ]:
!python src/colab_sync.py save --what models results

## 9. Where things stand

In [ ]:
!python src/colab_sync.py status
print()
!ls -la models/ results/figures/

---

Send the output of the preprocessing and training cells back to Claude Code, and
Phase 5 (evaluation: confusion matrices, the MobileNetV2 vs ResNet50 comparison,
efficiency metrics and Grad-CAM) can be built against the real results.